In [2]:
from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_chroma import Chroma 
import os
from dotenv import load_dotenv
from langchain.chat_models import init_chat_model


In [3]:
documents = PyPDFLoader("arquivos/Flutter_for_Beginners_by_Alessandro_Biessek_(z-lib.org).pdf").load()
print(f"Documento carregado. Total de documents: {len(documents)}")

Documento carregado. Total de documents: 498


In [6]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    length_function=len
)

docs = text_splitter.split_documents(documents)
print(f"total de chunks criados: {len(docs)}")

total de chunks criados: 972


In [7]:
persist_directory = "./chroma_db"
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

if os.path.exists(persist_directory):
    vector_store = Chroma(
        persist_directory=persist_directory, 
        embedding_function=embeddings
        )
    print("já criada")
else:
    vector_store = Chroma.from_documents(
        docs, 
        embeddings, 
        persist_directory=persist_directory
        )
    print("Não existe ainda. Criando Vector Store")
print("Vector Store criada e carregada!")


Não existe ainda. Criando Vector Store
Vector Store criada e carregada!


In [8]:
retriver = vector_store.as_retriever(search_kwargs={"k":2})
llm = init_chat_model(
    model="gemini-3.5-flash-lite", 
    model_provider="google_genai", 
    temperature=0
)

In [16]:
pergunta = input("Sou o oráculo do Flutter. Faça sua pergunta: ")
similar_docs = retriver.invoke(pergunta)
contexto = "\n\n".join([doc.page_content for doc in similar_docs])
resposta = llm.invoke(f"com base no seguinte contexto \n\n {contexto} \n\n responda a seguinte pergunta \n\n {pergunta}")

/home/jady/Documents/ESTUDOS_CODING/Langchain-course/Langchain-project/.venv/lib/python3.13/site-packages/langchain_google_genai/chat_models.py:3120: UserWarning: Model 'gemini-3.5-flash-lite' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


In [17]:
print(resposta.content[0]["text"])

Com base no contexto fornecido, **não é mencionada a versão específica do Flutter** (por exemplo, Flutter 2, 3, etc.). O texto apenas apresenta a introdução ao framework, os capítulos do livro (como os fundamentos de Dart, introdução ao Flutter, widgets e manipulação de entrada de usuário) e tópicos do sumário, mas em nenhum momento o número da versão do Flutter é citado.
